# Phase 1 — Zero-shot Baseline (v0)

**Mục đích:** Đo khả năng ước giá của `Qwen/Qwen3.5-4B-Base` CHƯA fine-tune trên 500 test samples.  
Kết quả cung cấp lower bound (v0) để đo improvement sau các phase training.

**Config chốt từ Phase 0:**
- Model: `Qwen/Qwen3.5-4B-Base` — Base (không Instruct), 4-bit NF4
- Dataset: `SeanSunny/items_prompts_tv_3` — test split, 500 random samples (seed=42)
- `max_seq_length = 192` | `max_new_tokens = 4`
- `pred_vnd = predict(prompt) * 1000` (completion là đơn vị nghìn đồng)
- Output: `results/v0_results.json`

**Kỳ vọng:** RMSLE > 1.0 — model chưa biết pattern completion số thuần. Đây là expected.

## 0. Cài đặt môi trường (chạy trên GPU rental)

**Môi trường đã xác nhận:** CUDA 12.8 | PyTorch 2.9.0+cu128 | RTX 3090 Ti (25.3 GB) | Compute 8.6

**Bước 1 — uv sync (sau khi git clone):**
```bash
uv sync
```

**Bước 2 — Cài Unsloth (chạy cell bên dưới):**
```bash
uv add unsloth
```
Unsloth 2025+ tự detect CUDA version. Nếu lỗi "no matching distribution", thử:
```bash
uv add "unsloth[cu128-torch290]"
```
Hoặc fallback pip:
```bash
pip install "unsloth[cu128-torch290]"
```

**Fallback hoàn toàn nếu Unsloth không cài được — HF transformers + bitsandbytes (chậm hơn ~2x):**
```bash
uv add bitsandbytes
```
Sau đó đổi `USE_UNSLOTH = False` trong cell Constants.

In [ ]:
# Chay cell nay truoc tien tren may thue GPU.
# Unsloth 2025+ tu detect CUDA — thu don gian truoc:
# !uv add unsloth

# Neu loi "no matching distribution", uncomment dong cu the hon:
# !uv add "unsloth[cu128-torch290]"

# Fallback hoan toan (HF transformers + bitsandbytes, cham hon 2x):
# !uv add bitsandbytes

In [ ]:
import os
import re
import sys
import json
import time
import math
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
from datasets import load_dataset
from huggingface_hub import login
from sklearn.metrics import mean_absolute_error, r2_score

# Add fine_tune_qwen/ to path de import utils
NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))

print("Imports OK")

In [ ]:
# Constants — chinh sua neu can

BASE_MODEL     = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME   = "SeanSunny/items_prompts_tv_3"
HF_USER        = "SeanSunny"

MAX_SEQ_LENGTH = 192
MAX_NEW_TOKENS = 4
EVAL_SAMPLES   = 500
SEED           = 42

USE_UNSLOTH    = True   # Doi thanh False neu Unsloth khong cai duoc

RESULTS_DIR    = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_FILE   = RESULTS_DIR / "v0_results.json"

print(f"BASE_MODEL     : {BASE_MODEL}")
print(f"DATASET_NAME   : {DATASET_NAME}")
print(f"MAX_SEQ_LENGTH : {MAX_SEQ_LENGTH}")
print(f"MAX_NEW_TOKENS : {MAX_NEW_TOKENS}")
print(f"EVAL_SAMPLES   : {EVAL_SAMPLES}")
print(f"USE_UNSLOTH    : {USE_UNSLOTH}")
print(f"RESULTS_FILE   : {RESULTS_FILE}")

In [ ]:
# GPU check

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM total     : {total_vram:.1f} GB")
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
    print(f"Compute cap    : {capability} -> bf16={'yes' if use_bf16 else 'no (fp16)'}")
else:
    raise RuntimeError("GPU khong kha dung. Kiem tra lai moi truong.")

In [ ]:
# HF Login — load tu tech2ai/.env (co san tren may thue)
from dotenv import load_dotenv

# .env nam o tech2ai/, notebook nam o fine_tune_qwen/ -> ../
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (loaded from {env_path})")
else:
    print(f"HF_TOKEN khong tim thay trong {env_path}")
    login()  # Interactive fallback

## 1. Load model Qwen3.5-4B-Base (4-bit)

In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,          # auto: bf16 neu GPU ho tro, fp16 neu khong
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)  # 2x faster inference

else:
    # Fallback: HF transformers + bitsandbytes
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
    )
    model.eval()

# Tokenizer fix chuan — padding = eos, side = right (giong English reference)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 2. Load dataset — 500 random test samples

In [ ]:
dataset = load_dataset(DATASET_NAME)
test_full = dataset["test"]

# 500 random samples voi seed=42
test_sample = test_full.shuffle(seed=SEED).select(range(EVAL_SAMPLES))

print(f"Test full      : {len(test_full):,} items")
print(f"Eval sample    : {len(test_sample):,} items")
print()
print("Sample item:")
print(f"  prompt[:80] : {test_sample[0]['prompt'][:80]!r}")
print(f"  completion  : {test_sample[0]['completion']!r}")
print(f"  price_vnd   : {test_sample[0]['price_vnd_true']:,}")

## 3. Predict function

Phiên bản đơn giản cho zero-shot baseline — chưa dùng StoppingCriteria.  
StoppingCriteria (3-layer inference safety) sẽ implement từ Phase 2 khi có fine-tuned model.  
Xem plan_day5.md Section 12 R8 để biết chi tiết rủi ro digit-by-digit.

In [ ]:
def predict_one(prompt: str) -> tuple[int, str]:
    """Returns (pred_thousands_vnd, raw_generated_text)."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy — deterministic
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)

    match = re.search(r"\d+", raw)
    pred = int(match.group()) if match else 0
    return pred, raw


# Chay thu 1 sample de kiem tra
sample_prompt = test_sample[0]["prompt"]
t0 = time.time()
pred_k, raw = predict_one(sample_prompt)
elapsed = time.time() - t0

print(f"Raw output     : {raw!r}")
print(f"pred_thousands : {pred_k}")
print(f"pred_vnd       : {pred_k * 1000:,}")
print(f"true_vnd       : {test_sample[0]['price_vnd_true']:,}")
print(f"Time/item      : {elapsed:.2f}s")

## 4. Inference — 500 samples

In [ ]:
preds_vnd  = []
trues_vnd  = []
raw_outputs = []

t_start = time.time()

for item in tqdm(test_sample, desc="Zero-shot inference"):
    pred_k, raw = predict_one(item["prompt"])
    preds_vnd.append(pred_k * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outputs.append(raw)

t_total = time.time() - t_start
print(f"\nDone: {EVAL_SAMPLES} samples in {t_total:.1f}s ({t_total/EVAL_SAMPLES:.2f}s/item)")
print(f"Items where model returned 0 (no digit found): {preds_vnd.count(0)}")

## 5. Compute metrics

In [ ]:
def rmsle(y_true, y_pred) -> float:
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


y_true = np.array(trues_vnd, dtype=float)
y_pred = np.array(preds_vnd, dtype=float)

metrics = {
    "rmsle" : rmsle(y_true, y_pred),
    "mae"   : float(mean_absolute_error(y_true, y_pred)),
    "mape"  : float(np.mean(np.abs(y_pred - y_true) / y_true) * 100),
    "r2"    : float(r2_score(y_true, y_pred)),
}

print("=" * 40)
print(f"v0 Zero-shot — {EVAL_SAMPLES} test samples")
print("=" * 40)
print(f"RMSLE : {metrics['rmsle']:.4f}  (primary, lower is better)")
print(f"MAE   : {metrics['mae']:,.0f} VND")
print(f"MAPE  : {metrics['mape']:.1f}%")
print(f"R2    : {metrics['r2']:.4f}")
print("=" * 40)
print("Baseline v8 (Day 4 reference): RMSLE=0.4004")
print(f"Gap v0 vs v8: {metrics['rmsle'] - 0.4004:+.4f}")

## 6. Save results

In [ ]:
# 20 sample predictions de kiem tra thu cong
samples_out = []
for i in range(20):
    true_vnd = trues_vnd[i]
    pred_vnd = preds_vnd[i]
    error_pct = abs(pred_vnd - true_vnd) / true_vnd * 100 if true_vnd > 0 else None
    samples_out.append({
        "idx"          : i,
        "prompt_excerpt": test_sample[i]["prompt"][:120],
        "generated_raw" : raw_outputs[i],
        "pred_vnd"      : pred_vnd,
        "true_vnd"      : true_vnd,
        "error_pct"     : round(error_pct, 1) if error_pct is not None else None,
    })

results = {
    "version"       : "v0_zero_shot",
    "model"         : BASE_MODEL,
    "dataset"       : DATASET_NAME,
    "eval_samples"  : EVAL_SAMPLES,
    "seed"          : SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "inference_sec" : round(t_total, 1),
    "sec_per_item"  : round(t_total / EVAL_SAMPLES, 2),
    "zero_pred_count": preds_vnd.count(0),
    "metrics"       : metrics,
    "samples"       : samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved: {RESULTS_FILE}")

In [ ]:
# Hien thi 20 samples
print(f"{'#':>3}  {'true_vnd':>12}  {'pred_vnd':>12}  {'err%':>8}  generated_raw")
print("-" * 75)
for s in samples_out:
    err_str = f"{s['error_pct']:>7.1f}%" if s["error_pct"] is not None else "    N/A"
    raw_repr = repr(s["generated_raw"])[:20]
    print(f"{s['idx']:>3}  {s['true_vnd']:>12,}  {s['pred_vnd']:>12,}  {err_str}  {raw_repr}")

## Tóm tắt kết quả v0

Chạy cell bên dưới để in leaderboard hiện tại sau khi có kết quả.

In [ ]:
print("Leaderboard Day 5 (cap nhat sau moi phase):")
print(f"{'Version':<20} {'RMSLE':>8} {'MAE':>12} {'MAPE':>8} {'R2':>8}")
print("-" * 60)
print(f"{'v0 zero-shot':<20} {metrics['rmsle']:>8.4f} {metrics['mae']:>12,.0f} {metrics['mape']:>7.1f}% {metrics['r2']:>8.4f}")
print(f"{'v8 Day4 (ref)':<20} {'0.4004':>8} {'79,853':>12} {'30.7%':>8} {'0.692':>8}")
print()
print("Buoc tiep: Phase 2 — Smoke test v1 (03_train_v1_smoke.ipynb)")
print("  Confirm voi user truoc khi chay.")